In [1]:
import torch
print(torch.__version__)

2.8.0+cu128


In [2]:
#音频预处理
import librosa
import numpy as np

def load_audio(path, sr=48000, duration=10):
    audio, _ = librosa.load(path, sr=sr, mono=True)

    # 截断 / padding
    target_len = sr * duration
    if len(audio) > target_len:
        audio = audio[:target_len]
    else:
        pad = target_len - len(audio)
        audio = np.pad(audio, (0, pad))

    return audio

In [3]:
#得到manifest
import pandas as pd
import json
import os

CSV_PATH = "/root/autodl-tmp/ESC-50/meta/esc50.csv"
AUDIO_DIR = "data/raw_audio/esc50"

df = pd.read_csv(CSV_PATH)

manifest = []

for _, row in df.iterrows():
    path = os.path.join(AUDIO_DIR, row["filename"])

    manifest.append({
        "path": path,
        "label": row["category"],
        "fold": int(row["fold"])
    })

with open("data/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

In [8]:
#模型封装
import torch
from transformers import AutoModel, AutoTokenizer

class CLAPWrapper:
    def __init__(self, device="cuda"):
        self.device = device
        self.model = AutoModel.from_pretrained("/root/autodl-tmp/clap-model").to(device)
        self.tokenizer = AutoTokenizer.from_pretrained("/root/autodl-tmp/clap-model")

    def get_text_embedding(self, text):
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        with torch.no_grad():
            emb = self.model.get_text_features(**inputs)
        return emb.cpu().numpy()

    def get_audio_embedding(self, audio):
        audio = torch.tensor(audio).unsqueeze(0).to(self.device)
        with torch.no_grad():
            emb = self.model.get_audio_features(audio)
        return emb.cpu().numpy()

In [6]:
import faiss
import numpy as np
import pickle

class FaissIndex:
    def __init__(self, dim):
        self.index = faiss.IndexFlatIP(dim)  # cosine用inner product
        self.metadata = []

    def add(self, vectors, metas):
        self.index.add(vectors)
        self.metadata.extend(metas)

    def search(self, query, k=5):
        D, I = self.index.search(query, k)
        results = []
        for idx, score in zip(I[0], D[0]):
            results.append({
                "score": float(score),
                "meta": self.metadata[idx]
            })
        return results

    def save(self, path):
        faiss.write_index(self.index, path + ".index")
        with open(path + ".meta", "wb") as f:
            pickle.dump(self.metadata, f)

    def load(self, path):
        self.index = faiss.read_index(path + ".index")
        with open(path + ".meta", "rb") as f:
            self.metadata = pickle.load(f)

In [ ]:
import os
import numpy as np
from tqdm import tqdm
from pathlib import Path

AUDIO_DIR = "data/raw_audio"
SAVE_PATH = "data/embeddings/audio_index"

def build():
    model = CLAPWrapper(device="cuda")
    vectors = []
    metas = []
    # 使用 pathlib 递归查找所有 .wav 文件
    audio_path_obj = Path(AUDIO_DIR)
    # rglob("*") 会递归查找所有子目录，.is_file() 确保排除文件夹
    files = [p for p in audio_path_obj.rglob("*.wav") if p.is_file()]

    print(f"共找到 {len(files)} 个音频文件")
    

    for path_obj in tqdm(files):
        path = str(path_obj)  # 转回字符串路径供 load_audio 使用

        try:
            audio = load_audio(path)
            emb = model.get_audio_embedding(audio)

            vectors.append(emb[0])
            metas.append({"path": path})

        except Exception as e:
            print(f"skip {f}: {e}")

    vectors = np.array(vectors).astype("float32")

    index = FaissIndex(dim=vectors.shape[1])
    index.add(vectors, metas)
    index.save(SAVE_PATH)

if __name__ == "__main__":
    build()

In [ ]:


INDEX_PATH = "data/embeddings/audio_index"

def search(query):
    model = CLAPWrapper(device="cuda")

    index = FaissIndex(dim=512)
    index.load(INDEX_PATH)

    q_emb = model.get_text_embedding(query)

    results = index.search(q_emb, k=5)

    for r in results:
        print(r)

if __name__ == "__main__":
    search("ocean waves relaxing")

In [20]:
import os
import torch
import faiss
import pickle
import librosa
import numpy as np
from tqdm import tqdm
from pathlib import Path
from transformers import AutoModel, AutoTokenizer

# ==========================================
# 1. 向量数据库封装 (Faiss)
# ==========================================
class FaissIndex:
    def __init__(self, dim):
        # IndexFlatIP 用于内积计算，配合 L2 归一化即为余弦相似度
        self.index = faiss.IndexFlatIP(dim)
        self.metadata = []

    def add(self, vectors, metas):
        # 显式转换确保类型正确
        vectors = np.ascontiguousarray(vectors).astype("float32")
        self.index.add(vectors)
        self.metadata.extend(metas)

    def save(self, path):
        faiss.write_index(self.index, f"{path}.index")
        with open(f"{path}.meta", "wb") as f:
            pickle.dump(self.metadata, f)

    def load(self, path):
        self.index = faiss.read_index(f"{path}.index")
        with open(f"{path}.meta", "rb") as f:
            self.metadata = pickle.load(f)

# ==========================================
# 2. CLAP 模型封装
# ==========================================
from transformers import ClapProcessor, ClapModel

class CLAPWrapper:
    def __init__(self, model_path, device="cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        self.model = ClapModel.from_pretrained(model_path).to(self.device)
        self.processor = ClapProcessor.from_pretrained(model_path)

    def get_audio_embedding(self, audio):
        # 使用官方 Processor 处理音频，它会自动处理采样率、对齐和特征提取
        # 注意：sampling_rate 必须与加载音频时一致
        inputs = self.processor(
            audio=[audio],
            return_tensors="pt",
            sampling_rate=48000
        ).to(self.device)
        
        with torch.no_grad():
            # 此时使用 model 直接处理封装后的 inputs
            outputs = self.model.get_audio_features(**inputs)
            emb = outputs.pooler_output.cpu().numpy()
        return emb
    
    def get_text_embedding(self, text):
        inputs = self.processor(
            text=[text],
            return_tensors="pt",
            padding=True
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model.get_text_features(**inputs)
            emb = outputs.pooler_output.cpu().numpy()

        return emb

# ==========================================
# 3. 工具函数
# ==========================================
def load_audio(path, sr=48000, max_duration=10):
    y, _ = librosa.load(path, sr=sr, mono=True)

    max_len = sr * max_duration
    if len(y) > max_len:
        y = y[:max_len]
    elif len(y) < max_len:
        y = np.pad(y, (0, max_len - len(y)))

    return y

# ==========================================
# 4. 主构建逻辑
# ==========================================
# 配置参数
AUDIO_DIR = "data/raw_audio"
MODEL_PATH = "/root/autodl-tmp/clap-model"
SAVE_PATH = "data/embeddings/audio_index"

def build():
    # 环境检查
    if not os.path.exists(MODEL_PATH):
        print(f"错误: 找不到模型路径 {MODEL_PATH}")
        return

    # 初始化模型
    model = CLAPWrapper(MODEL_PATH)
    vectors = []
    metas = []

    # 递归查找所有 .wav 文件，排除 .json 或其他杂质
    audio_path_obj = Path(AUDIO_DIR)
    files = [p for p in audio_path_obj.rglob("*.wav") if p.is_file()]
    
    if not files:
        print(f"在 {AUDIO_DIR} 中没有找到任何 .wav 文件！")
        return

    print(f"开始处理 {len(files)} 个音频文件...")

    for path_obj in tqdm(files):
        path_str = str(path_obj)
        try:
            # 加载并提取
            audio = load_audio(path_str)
            emb = model.get_audio_embedding(audio) # 形状 (1, D)

            # 关键：确保是一维并存入
            v = emb[0].astype("float32")
            vectors.append(v)
            metas.append({"path": path_str})

        except Exception as e:
            print(f"\n[跳过] 处理 {path_str} 时出错: {e}")

    # 汇总数据
    if len(vectors) == 0:
        print("没有提取到任何有效向量。")
        return

    # 转换为 numpy 矩阵 (N, dim)
    vectors_np = np.stack(vectors)
    
    # 强制进行 L2 归一化（使 IndexFlatIP 等同于余弦相似度）
    faiss.normalize_L2(vectors_np)

    print(f"构建索引中，向量形状: {vectors_np.shape}")

    # 创建并保存索引
    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
    index = FaissIndex(dim=vectors_np.shape[1])
    index.add(vectors_np, metas)
    index.save(SAVE_PATH)

    print(f"🎉 任务完成！索引已保存至: {SAVE_PATH}.index")



In [ ]:
if __name__ == "__main__":
    build()

In [21]:

import faiss
import numpy as np

INDEX_PATH = "data/embeddings/audio_index"
MODEL_PATH = "/root/autodl-tmp/clap-model"

def search(query, topk=5):
    model = CLAPWrapper(MODEL_PATH)

    index = FaissIndex(dim=512)
    index.load(INDEX_PATH)

    # 文本 embedding
    q_emb = model.get_text_embedding(query)

    # 归一化（必须和建库一致）
    faiss.normalize_L2(q_emb)

    # 检索
    D, I = index.index.search(q_emb.astype("float32"), topk)

    results = []
    for idx, score in zip(I[0], D[0]):
        meta = index.metadata[idx]
        results.append({
            "score": float(score),
            "path": meta["path"]
        })

    return results


if __name__ == "__main__":
    res = search("ocean waves relaxing")

    for r in res:
        print(r)

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

{'score': 0.5318350791931152, 'path': 'data/raw_audio/esc50/3-187710-A-11.wav'}
{'score': 0.5250780582427979, 'path': 'data/raw_audio/esc50/4-195497-B-11.wav'}
{'score': 0.507968544960022, 'path': 'data/raw_audio/esc50/3-164120-A-11.wav'}
{'score': 0.5061087012290955, 'path': 'data/raw_audio/esc50/2-137162-A-11.wav'}
{'score': 0.5052155256271362, 'path': 'data/raw_audio/esc50/4-182613-B-11.wav'}
